# 🔀 Módulo 14 — Workflows: Edges Condicionales

> **Objetivo:** Crear bifurcaciones en el grafo del workflow basadas en el contenido del mensaje.

## ¿Qué es un edge condicional?

Un edge (arista) entre dos ejecutores que **sólo se activa si una condición se cumple**.

```
                       ┌─→ [PositiveHandler]   (si is_positive == True)
[SentimentAnalyzer] ───┤
                       └─→ [NegativeHandler]   (si is_positive == False)
```

## API

```python
.add_edge(from_node, to_node, condition=lambda msg: ...)
```

La función `condition` recibe el mensaje producido por `from_node`. Si devuelve `True`,
el edge se activa y el mensaje llega a `to_node`.

> 🎉 **No requiere Azure OpenAI** — los ejemplos son determinísticos.


In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from dataclasses import dataclass
from typing_extensions import Never
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, executor, handler

print("✅ Listo")


## 1️⃣ Edges directos en cadena (sin condición)

Para repaso: `add_edge(A, B)` sin `condition` crea un edge **incondicional**.


In [ ]:
@executor(id="uppercase")
async def uppercase(text: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(text.upper())


@executor(id="formatter")
async def formatter(text: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(f"[{text}]")


@executor(id="final_output")
async def final_output(text: str, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(f"Final: {text}")


workflow = (
    WorkflowBuilder(start_executor=uppercase)
    .add_edge(uppercase, formatter)
    .add_edge(formatter, final_output)
    .build()
)

events = await workflow.run("test data")
print(f"✅ Cadena directa → {events.get_outputs()}")


## 2️⃣ Edge condicional — bifurcación según sentimiento

El analyzer produce un `SentimentResult`. Dos edges con condiciones opuestas enrutan al handler correcto.

> Las condiciones son funciones puras `lambda msg: bool` — no requieren AI, ejecutables localmente.


In [ ]:
from dataclasses import dataclass


@dataclass
class SentimentResult:
    is_positive: bool
    text: str


_POSITIVE_WORDS = {"good", "great", "excellent", "happy", "love", "wonderful", "amazing"}


class SentimentAnalyzerExecutor(Executor):
    def __init__(self, id: str = "sentiment_analyzer"):
        super().__init__(id=id)

    @handler
    async def analyze(self, text: str, ctx: WorkflowContext[SentimentResult]) -> None:
        is_positive = any(w in text.lower() for w in _POSITIVE_WORDS)
        await ctx.send_message(SentimentResult(is_positive=is_positive, text=text))


@executor(id="positive_handler")
async def positive_handler(msg: SentimentResult, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(f"POSITIVE: {msg.text}")


@executor(id="negative_handler")
async def negative_handler(msg: SentimentResult, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(f"NEGATIVE: {msg.text}")


# --- Workflow positivo ---
analyzer = SentimentAnalyzerExecutor()

workflow_pos = (
    WorkflowBuilder(start_executor=analyzer)
    .add_edge(
        analyzer,
        positive_handler,
        condition=lambda msg: isinstance(msg, SentimentResult) and msg.is_positive,
    )
    .add_edge(
        analyzer,
        negative_handler,
        condition=lambda msg: isinstance(msg, SentimentResult) and not msg.is_positive,
    )
    .build()
)

events = await workflow_pos.run("This is a great day!")
print(f"✅ Input positivo  → {events.get_outputs()}")


## 3️⃣ Mismo workflow, input negativo → el otro camino

Verificamos que el edge condicional opuesto se active correctamente.


In [ ]:
# Reconstruimos el workflow con instancias frescas para el segundo run
analyzer2 = SentimentAnalyzerExecutor(id="sentiment_analyzer_2")

workflow_neg = (
    WorkflowBuilder(start_executor=analyzer2)
    .add_edge(
        analyzer2,
        positive_handler,
        condition=lambda msg: isinstance(msg, SentimentResult) and msg.is_positive,
    )
    .add_edge(
        analyzer2,
        negative_handler,
        condition=lambda msg: isinstance(msg, SentimentResult) and not msg.is_positive,
    )
    .build()
)

events = await workflow_neg.run("This is terrible and awful")
print(f"✅ Input negativo  → {events.get_outputs()}")


## 🎯 Resumen

| Concepto | API |
|----------|-----|
| Edge directo | `.add_edge(A, B)` |
| Edge condicional | `.add_edge(A, B, condition=lambda msg: ...)` |
| Múltiples salidas | Varios ejecutores pueden llamar `yield_output` — todos terminales |

> 💡 Las condiciones son **funciones Python normales** — puedes usar cualquier lógica
> (incluyendo llamadas síncronas a APIs, comprobaciones de tipos, etc.).

**Siguiente módulo →** [Módulo 15 — Workflows: Eventos y Loops](./15_workflows_events.ipynb)
